# Семинар 7. Прогнозирование временных рядов

**Цель семинара:** Обучиться методам анализа и прогнозирования бизнес-показателей, развернутых во времени (динамика продаж, операционный спрос, трафик). Вы освоите классическую декомпозицию ряда на тренд и сезонность, построение прогнозных моделей с помощью Meta Prophet и сравнение их с классическими авторегрессионными моделями (ARIMA). Итогом станет production-функция `forecast_trends` для специализированного модуля `src/forecasting.py`.

### 🔧 Настройка окружения и импорт библиотек

Для работы с временными рядами нам потребуются статистические библиотеки `statsmodels` (для ARIMA и декомпозиции) и `prophet` (для продвинутого прогнозирования). Убедитесь, что они установлены (`uv add statsmodels prophet`).


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_error, mean_squared_error

try:
    from prophet import Prophet
except ImportError:
    print("⚠️ Установите пакет prophet: uv add prophet")

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)


---

## 📥 Шаг 1. Инициализация локального контура (Временной трек)

В отличие от задач классификации, для временных рядов нам нужна агрегация по дате (например, общая сумма транзакций в день). Мы сгенерируем временной трек, имитирующий операционные данные с трендом и еженедельной сезонностью.


In [ ]:
INPUT_DIR = os.path.join(".", "data", "processed")
ts_csv_path = os.path.abspath(os.path.join(INPUT_DIR, "time_series_data.csv"))

if not os.path.exists(ts_csv_path):
    print(f"⚠️ Файл временных рядов не найден. ")
    os.makedirs(INPUT_DIR, exist_ok=True)
    
df_ts = pd.read_csv(ts_csv_path)
df_ts['Date'] = pd.to_datetime(df_ts['Date'])
print(f"Временной ряд успешно загружен. Размерность: {df_ts.shape}")


---

## 🛠 ЗАДАНИЕ 1: Классическая статистическая декомпозиция
**Бизнес-контекст:** Любой бизнес-показатель во времени состоит из трех компонентов:
* **Тренд:** Долгосрочное направление (растем или падаем).
* **Сезонность:** Циклические колебания (например, всплески продаж по выходным).
* **Остаток (Шум):** Случайные события, которые нельзя предсказать.
Декомпозиция позволяет руководству увидеть чистый тренд, очищенный от сезонных скачков.

**Инструкция (TODO):**
1. Установите колонку с датой (`Date`) в качестве индекса датафрейма.
2. Используйте функцию `seasonal_decompose` из `statsmodels`. Передайте колонку со значениями (`Value`), укажите `model='additive'` и период `period=7` (недельная сезонность).
3. Постройте график результатов с помощью метода `.plot()`.

*🤖 Теги для AI-ментора: #SEM7_TASK1_START, #SEM7_TASK1_BUG*


In [ ]:
# [MASTER SOLUTION]
# Подготовка данных (индекс - дата)
ts_data = df_ts.set_index('Date')['Value']

# Декомпозиция
decomposition = seasonal_decompose(ts_data, model='additive', period=7)

# Визуализация
fig = decomposition.plot()
fig.set_size_inches(10, 8)
plt.tight_layout()
plt.show()


In [ ]:
# [STUDENT TEMPLATE]
# TODO: 1.1. Сделайте колонку 'Date' индексом и выберите колонку 'Value'
# TODO: 1.2. Примените seasonal_decompose с параметрами model='additive' и period=7
# TODO: 1.3. Визуализируйте результат
raise NotImplementedError("Задание 1 не выполнено! Удалите эту строку и напишите свой код.")

ts_data = df_ts.set_index(...)['...']
decomposition = seasonal_decompose(..., model='additive', period=7)

fig = decomposition.plot()
fig.set_size_inches(10, 8)
plt.show()


---

## 🛠 ЗАДАНИЕ 2: Прогнозирование с помощью Meta Prophet
**Бизнес-контекст:** Prophet — это индустриальный стандарт от Meta, который отлично справляется с бизнес-данными. Он автоматически учитывает праздники, смены трендов и пропущенные значения, а также строит доверительные интервалы, что критически важно для риск-менеджмента (оценка пессимистичного и оптимистичного сценариев).

**Инструкция (TODO):**
1. Prophet требует строгого формата названий колонок: дата должна называться `ds`, а значение — `y`. Переименуйте колонки.
2. Инициализируйте объект `Prophet(yearly_seasonality=False, daily_seasonality=False)`.
3. Обучите модель (`.fit()`) на подготовленном датафрейме.
4. Создайте датафрейм для будущих дат (`make_future_dataframe(periods=30)`).
5. Сделайте прогноз (`.predict()`) и постройте график (`.plot()`).

*🤖 Теги для AI-ментора: #SEM7_TASK2_START, #SEM7_TASK2_BUG*


In [ ]:
# [MASTER SOLUTION]
# 1. Подготовка формата Prophet
df_prophet = df_ts.rename(columns={'Date': 'ds', 'Value': 'y'})

# 2. Инициализация и обучение
m = Prophet(yearly_seasonality=False, daily_seasonality=False)
m.fit(df_prophet)

# 3. Создание горизонта прогнозирования (30 дней)
future = m.make_future_dataframe(periods=30, freq='D')

# 4. Предсказание
forecast = m.predict(future)

# 5. Визуализация
fig1 = m.plot(forecast)
plt.title("Прогноз Prophet (с доверительными интервалами)")
plt.xlabel("Дата")
plt.ylabel("Значение")
plt.show()


In [ ]:
# [STUDENT TEMPLATE]
# TODO: 2.1. Создайте df_prophet, переименовав 'Date' во 'ds' и 'Value' в 'y'
# TODO: 2.2. Инициализируйте Prophet() и вызовите .fit(df_prophet)
# TODO: 2.3. Создайте future-датафрейм на 30 периодов
# TODO: 2.4. Вызовите .predict(future) и визуализируйте через .plot()
raise NotImplementedError("Задание 2 не выполнено! Удалите эту строку и напишите свой код.")

df_prophet = df_ts.rename(columns={...})

m = Prophet()
m.fit(...)

future = m.make_future_dataframe(periods=30, freq='D')
forecast = m.predict(...)

m.plot(forecast)
plt.show()


---

## 🛠 ЗАДАНИЕ 3: Оценка точности и сравнение с классикой (ARIMA)
**Бизнес-контекст:** Чтобы понять, можно ли доверять прогнозу, мы рассчитываем метрики ошибки на исторической части.
* **MAE:** Средняя абсолютная ошибка (на сколько единиц мы ошибаемся в среднем).
* **MAPE:** Средняя абсолютная ошибка в процентах (понятно бизнесу).

**Инструкция (TODO):**
1. Извлеките исторические прогнозы из датафрейма `forecast` (они находятся в колонке `yhat`).
2. Рассчитайте `MAE` между фактическими значениями (`df_prophet['y']`) и предсказанными (`yhat`).
3. Для сравнения обучите простейшую модель `ARIMA` из `statsmodels` с порядком `order=(5,1,0)`.


In [ ]:
# [MASTER SOLUTION]
# 1. Метрики Prophet на исторических данных (In-sample)
historical_forecast = forecast.iloc[:len(df_prophet)]
y_true = df_prophet['y']
y_pred = historical_forecast['yhat']

mae_prophet = mean_absolute_error(y_true, y_pred)
rmse_prophet = np.sqrt(mean_squared_error(y_true, y_pred))
mape_prophet = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

print("--- Метрики Prophet ---")
print(f"MAE:  {mae_prophet:.2f}")
print(f"RMSE: {rmse_prophet:.2f}")
print(f"MAPE: {mape_prophet:.2f}%\n")

# 2. Сравнение с ARIMA
print("⏳ Обучение ARIMA(5,1,0)...")
arima_model = ARIMA(df_ts['Value'], order=(5, 1, 0))
arima_result = arima_model.fit()

# Предсказание In-sample
arima_pred = arima_result.predict(start=0, end=len(df_ts)-1)
mae_arima = mean_absolute_error(df_ts['Value'], arima_pred)

print("--- Метрики ARIMA ---")
print(f"MAE:  {mae_arima:.2f}")

# Вывод: Prophet обычно лучше улавливает сложную сезонность без сложной ручной настройки.


In [ ]:
# [STUDENT TEMPLATE]
# TODO: 3.1. Рассчитайте MAE и RMSE для Prophet (используйте функции из sklearn.metrics)
# TODO: 3.2. Обучите ARIMA(df_ts['Value'], order=(5,1,0)) и вызовите .fit()
raise NotImplementedError("Задание 3 не выполнено! Удалите эту строку и напишите свой код.")

y_true = df_prophet['y']
y_pred = forecast.iloc[:len(df_prophet)]['yhat']

mae_prophet = mean_absolute_error(..., ...)
print(f"Prophet MAE: {mae_prophet:.2f}")

arima_model = ARIMA(..., order=(5, 1, 0))
arima_result = arima_model.fit()
# Дальнейший расчет метрик


---

## 🏗 ФИНАЛЬНАЯ СБОРКА: Сквозная функция forecast_trends

Согласно архитектуре курса, вся логика прогнозирования выносится в отдельный специализированный модуль ядра `src/forecasting.py`. Соберем параметризованную функцию `forecast_trends`.


In [ ]:
# [MASTER SOLUTION]
def forecast_trends(df: pd.DataFrame, date_col: str, value_col: str, periods: int, freq: str = 'D') -> pd.DataFrame:
    """
    Промышленная функция автоматического анализа динамики и генерации 
    предиктивных значений на заданный горизонт планирования (с доверительными интервалами).
    """
    # 1. Подготовка данных
    df_clean = df[[date_col, value_col]].copy()
    df_clean = df_clean.rename(columns={date_col: 'ds', value_col: 'y'})
    df_clean['ds'] = pd.to_datetime(df_clean['ds'])
    
    # 2. Инициализация и обучение
    # Отключаем логирование Prophet для чистоты вывода
    import logging
    logging.getLogger('prophet').setLevel(logging.WARNING)
    
    model = Prophet(
        yearly_seasonality=False,
        weekly_seasonality=True,
        daily_seasonality=False
    )
    model.fit(df_clean)
    
    # 3. Прогнозирование
    future = model.make_future_dataframe(periods=periods, freq=freq)
    forecast = model.predict(future)
    
    # 4. Форматирование результата
    # Возвращаем дату, прогноз, нижнюю и верхнюю границы (для пессимистичного/оптимистичного сценариев)
    result_df = forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].copy()
    result_df = result_df.rename(columns={'ds': date_col})
    
    return result_df

# Запуск пайплайна прогнозирования
forecast_df = forecast_trends(
    df=df_ts,
    date_col='Date',
    value_col='Value',
    periods=30,
    freq='D'
)
print("Пайплайн прогнозирования успешно завершен!")
display(forecast_df.tail(5))


In [ ]:
# [STUDENT TEMPLATE]
def forecast_trends(df: pd.DataFrame, date_col: str, value_col: str, periods: int, freq: str = 'D') -> pd.DataFrame:
    """
    Промышленная функция для генерации прогноза во времени.
    """
    # TODO: Соберите архитектуру функции, используя переменные date_col и value_col
    raise NotImplementedError("Финальная сборка функции не выполнена!")
    
    # Подготовка формата для Prophet (переименование во ds и y)
    ...
    
    # Инициализация Prophet и обучение
    ...
    
    # Создание future датафрейма и прогноз
    ...
    
    # Возврат датафрейма со столбцами: дата, yhat, yhat_lower, yhat_upper
    return pd.DataFrame()

forecast_df = forecast_trends(df_ts, 'Date', 'Value', 30)


---

## 🛠 Автоматизированная проверка качества (Autocheck)

Скрипт проверяет, возвращает ли функция прогнозирования корректную структуру данных с доверительными интервалами.


In [ ]:
def run_autocheck(forecast_result, original_df, periods):
    print("🚀 Тестирование функции forecast_trends...\n" + "-"*45)
    validation_status = True
    
    if not isinstance(forecast_result, pd.DataFrame) or forecast_result.empty:
        print("❌ Ошибка: Возвращен пустой или невалидный DataFrame.")
        return False
        
    expected_cols = ['Date', 'yhat', 'yhat_lower', 'yhat_upper']
    missing_cols = [c for c in expected_cols if c not in forecast_result.columns]
    
    # Проверка структуры столбцов
    if missing_cols:
        print(f"❌ Ошибка: В таблице отсутствуют обязательные столбцы: {missing_cols}")
        print("Убедитесь, что вы вернули прогноз и доверительные интервалы.")
        validation_status = False
    else:
        print("✅ Колонки прогноза и доверительных интервалов (yhat, yhat_lower, yhat_upper) присутствуют.")
        
    # Проверка горизонта прогноза
    expected_length = len(original_df) + periods
    actual_length = len(forecast_result)
    
    if actual_length != expected_length:
        print(f"❌ Ошибка: Неверная длина прогноза. Ожидалось {expected_length}, получено {actual_length}.")
        validation_status = False
    else:
        print(f"✅ Горизонт прогнозирования корректен (добавлено {periods} периодов).")
        
    # Проверка логики интервалов
    if 'yhat' in forecast_result.columns:
        invalid_intervals = forecast_result[
            (forecast_result['yhat'] < forecast_result['yhat_lower']) | 
            (forecast_result['yhat'] > forecast_result['yhat_upper'])
        ]
        if len(invalid_intervals) > 0:
            print("❌ Ошибка: Базовый прогноз выходит за рамки доверительных интервалов.")
            validation_status = False
        else:
            print("✅ Математическая логика доверительных интервалов соблюдена (lower <= yhat <= upper).")

    print("-" * 45)
    if validation_status:
        print("🎉 ПОЗДРАВЛЯЕМ! Модуль прогнозирования временных рядов готов.")
        print("Перенесите эту функцию в course_project/src/forecasting.py!")
    else:
        print("⚠️ Обнаружены технические дефекты. Проверьте реализацию вашей функции.")

run_autocheck(forecast_df, df_ts, 30)
